In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.impute import KNNImputer
from functools import reduce
import os
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None
from IPython.display import display
from pathlib import Path

In [ ]:
# Reset cells output
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')
pd.reset_option('display.width')
pd.reset_option('display.expand_frame_repr')

In [ ]:
print("hello")

In [ ]:
# Testing data accessibity

RAW_DIR = r"C:\Users\bouba.ismalia\Stichting Hogeschool Utrecht\FCA-DA-P - data"
sdo2_skc_path = Path(RAW_DIR) / "sdo2-skc.csv"

sdo2_skc_df = pd.read_csv(sdo2_skc_path, encoding="utf-8-sig", sep=";")
print(sdo2_skc_df.head())

In [ ]:
from pathlib import Path
import pandas as pd

# Base folder (no .env)
RAW_DIR = Path(r"C:\Users\bouba.ismalia\Stichting Hogeschool Utrecht\FCA-DA-P - data\Version 1 data")

# Excel file path
file_path = RAW_DIR / "ODS Osiris basistabellen Inschrijving etc.xlsx"

# Optional: sanity check
if not file_path.exists():
    raise FileNotFoundError(f"Not found: {file_path}")

# Load every sheet into a dict of DataFrames
data = pd.read_excel(file_path, sheet_name=None)  # engine inferred (xlsx -> openpyxl)

# Print the names of all sheets
print("Sheet names:", list(data.keys()))

# Display the first few rows of each sheet
for sheet_name, sheet_df in data.items():
    print(f"--- {sheet_name} ---")
    print(sheet_df.head())
    print()

# (Optional) If you want a specific sheet:
# students_df = pd.read_excel(file_path, sheet_name="Students")
# print(students_df.head())


In [ ]:
# Load environment variables
from dotenv import load_dotenv
import os
from pathlib import Path
import pandas as pd

load_dotenv()

# Directories from .env
RAW_DIR = Path(os.getenv("STUDENT_RAW_DIR")).expanduser()
OUT_DIR = Path(os.getenv("OUTPUTS_DIR")).expanduser()

for p in (RAW_DIR, OUT_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("RAW_DIR :", RAW_DIR)
print("OUT_DIR :", OUT_DIR)

# File path using RAW_DIR
file_path = RAW_DIR / "1.2 250319_DSP_StudentDropoutProject_DWH-OSI_combi.xlsx"

# Load the Excel file
data = pd.read_excel(file_path, sheet_name=None)

# Print the names of all sheets
print("Sheet names:", data.keys())

# Display the first few rows of each sheet
for sheet_name, sheet_data in data.items():
    print(f"---{sheet_name}---")
    print(sheet_data.head())


In [ ]:
# Load only the 'DATA' sheet into a DataFrame
data = pd.read_excel(file_path, sheet_name='DATA')

# Display the first few rows of the 'DATA' sheet
data

In [ ]:
data.columns

In [ ]:
# renaming columns for data
data_rename_mapping = {
    'studentnummer': 'Student_num',
    'sinh_d': 'SINH_ID',
    'cohort_opleiding': 'Study_Pro_Start_Year',
    'collegejaar': 'College_Year',
    'ingangsdatum': 'Student_Start_Date',
    'afloopdatum': 'Student_End_Date',
    'vorm': 'Student_Full_Parttime_Dual',
    'opl_naam_en': 'Study_Prog_Name',
    'opleiding': 'Study_Prog_Code',
    'croho': 'Study_Prog_Group_ID',
    'type_ho': 'Study_Bach_Master_Type',
    'fase': 'Study_Prog_Phase',
    'p_examendatum': 'Study_Prog_Exam_Date',
    'p_behaald': 'Study_Prog_Exam_Completed',
    'uit_zonder_dip': 'Dropped_No_Diploma',
    'opleiding_switch': 'Switched',
    'C1': 'Graduating',
    'C2': 'Dropping',
    'C3': 'Switching',
    'vooropleiding': 'Prior_Edu_ID',
    'type_vooropleiding': 'Prior_Edu_Type',
    'datum_eindexamen': 'Prior_Edu_Exam_Date',
    'school': 'Prior_Edu_School_Code',
    'naam': 'Prior_Edu_School_Name',
    'plaats': 'Prior_Edu_School_Location',
    'postcode': 'Prior_Edu_Postcode',
    'land': 'Prior_Edu_Country'
    }

# Rename columns
data.rename(columns=data_rename_mapping, inplace=True)
# Create a list of the new column names
new_columns = list(data_rename_mapping.values())
# Keep only the renamed columns and drop the rest, this is because we want to avoid the problem of a new column added in the database renders the tool not working. 
# A randomly added column is not cleaned and we should be notified of this to properly clean and add the new column.
data = data[new_columns]

In [ ]:
data.columns

In [ ]:
data

In [ ]:


# Ensure no two or more columns have the same values
identical_columns = []
columns = data.columns.tolist()

for i in range(len(columns)):
    for j in range(i+1, len(columns)):
        col1, col2 = columns[i], columns[j]
        if data[col1].equals(data[col2]):
            identical_columns.append((col1, col2))

if identical_columns:
    print("Identical columns found:")
    for col_pair in identical_columns:
        print(col_pair)
else:
    print("No identical columns found.")


In [ ]:
# Delete the duplicates and only leave columns whose names make more sense
del data['College_Year']
del data['Dropped_No_Diploma']
del data['Switched']

In [ ]:
data.Study_Pro_Start_Year.unique()

In [ ]:
# Delete IDs that do not add value to the analysis

# del data['Enrollment_ID']
del data['Study_Prog_Code']
del data['Prior_Edu_ID']
del data['Prior_Edu_School_Code']

In [ ]:
data.info()

In [ ]:
data.isnull().sum()

In [ ]:
#  For now, delete the Study_Prog_Exam_Date because it has more than 50% of missing values, because of this high percentage of missing values, imputing them can unreliable.
del data['Study_Prog_Exam_Date']

In [ ]:
data.columns

In [ ]:
#pd.set_option('display.max_rows', None)   # Show all rows (be careful if huge!)
#pd.set_option('display.max_columns', None)
#pd.set_option('display.max_colwidth', None)
#pd.set_option('display.width', 0)         # Automatically adjusts line-wrapping
#pd.set_option('display.expand_frame_repr', False)  

# Print value counts for all columns
for col in data.columns:
    print(f"Value counts for column: {col}")
    print(data[col].value_counts(dropna=False))  # include NaN values in the count
    print("\n" + "="*50 + "\n")

In [ ]:
data.columns

# Feature Engineering

In [ ]:
# Student_ID: repeats but Enrollment_ID is unique, focus on Enrollement_ID
# Student_Full_Parttime_Dual: has only one value, delete
# Study_Bach_Master_Type: has only one value, delete
# Study_Prog_Phase: has only one value, delete
# Study_Prog_Name: collapse all subjects of Teacher Education to one category "Teacher_Education". 
  # check if sentence contains the key words Teacher Education
  # Add underscore between the words in all categories to make it one word, for this check object data types.
# Study_Prog_Group_ID: Create study program group size from this
# Prior_Edu_Type: if value count < 200, replace with  others
# Prior_Edu_School_Name: if value count < 100, replace with  others
# Prior_Edu_Exam_Date:Create Student_Enrollment_Gap from this and Student_Start_Date
# Prior_Edu_School_Location: if value count < 100, replace with  others, reclassify these locations into provinces?
  # Use Prior_Edu_Postcode instead, its more reliable than Prior_Edu_School_Location



In [ ]:
# Student_Full_Parttime_Dual: has only one value, delete

del data['Student_Full_Parttime_Dual']
del data['Study_Bach_Master_Type']
del data['Study_Prog_Phase']

Engineer Study_Prog_Name

In [ ]:
print(data['Study_Prog_Name'].value_counts())

In [ ]:
def clean_study_prog_name(value):
    if isinstance(value, str):
        # 1. Remove leading "B " if present
        if value.startswith("B "):
            value = value[2:]
        
        # 2. Replace based on keywords
        if "Teacher" in value:
            value = "Teacher_Education"
        elif "Arts Therapies" in value:
            value = "Arts_Therapies"
        elif "Education in Primary Schools" in value:
            value = "Primary_Education"
        else:
            # 3. Remove "&" and "," signs
            value = value.replace("&", "").replace(",", "").strip()
        
        # 4. Replace spaces between words with underscores
        value = "_".join(value.split())
        return value
    return value

# Apply the function to the column
data['Study_Prog_Name'] = data['Study_Prog_Name'].apply(clean_study_prog_name)

In [ ]:
print(data['Study_Prog_Name'].value_counts())

Engineer Study_Prog_Group_ID

In [ ]:
print(data['Study_Prog_Group_ID'].value_counts())

In [ ]:
def add_group_size_column(data, group_column, new_column):
    """
    Adds a new column to the DataFrame that contains the count of each group based on the `group_column`.
    
    Parameters:
        data (DataFrame): The DataFrame where the new column will be added.
        group_column (str): The name of the column to group by.
        new_column (str): The name of the new column that will store the group sizes.
        
    Returns:
        DataFrame: The DataFrame with the new column added.
    """
    # Create a dictionary of counts for each group
    group_size = data[group_column].value_counts().to_dict()

    # Map the counts back to the dataframe as a new column
    data[new_column] = data[group_column].map(group_size)
    
    # Optionally drop the group column (if needed)
    del data[group_column]
    
    return data
    
# Example usage
data = add_group_size_column(data, 'Study_Prog_Group_ID', 'Study_Prog_Group_Size')

# Check if the new column is created and see its unique values
print(data['Study_Prog_Group_Size'].unique())


Engineer Prior_Edu_Type 

In [ ]:
print(data['Prior_Edu_Type'].value_counts())

In [ ]:
def clean_prior_edu_type(data, threshold=100, replacement="OVERIG"):
    """
    Replaces values in the 'Prior_Edu_Type' column that appear less than a specified threshold with a replacement value.
    
    Parameters:
        data (DataFrame): The DataFrame where the replacement will be applied.
        threshold (int): The threshold for the minimum count, below which values will be replaced.
        replacement (str): The value that will replace the smaller counts.
        
    Returns:
        DataFrame: The DataFrame with the modified 'Prior_Edu_Type' column.
    """
    # Get the value counts for the 'Prior_Edu_Type' column
    value_counts = data['Prior_Edu_Type'].value_counts()
    
    # Identify values with counts less than the threshold
    small_values = value_counts[value_counts < threshold].index
    
    # Replace values with counts less than the threshold with 'Others'
    data['Prior_Edu_Type'] = data['Prior_Edu_Type'].replace(small_values, replacement)
    
    return data
# Call the function to clean the 'Prior_Edu_Type' column
data = clean_prior_edu_type(data, threshold=100, replacement="OVERIG")

# Check the updated value counts
print(data['Prior_Edu_Type'].value_counts())


Engineer Prior_Edu_School_Name 

In [ ]:
print(data['Prior_Edu_School_Name'].value_counts())

In [ ]:
# Looking at how clumsy the sentences look like, Its better to use the Prior_Edu_Postcode column instead, later the postcode can be used to match the school name

Engineer Prior_Edu_Postcode

In [ ]:
print(data['Prior_Edu_Postcode'].value_counts())

In [ ]:
data.info()

In [ ]:
# Function to clean Prior_Edu_Postcode
def clean_prior_edu_postcode(data, threshold=50, replacement="Others"):
    """
    Cleans the 'Prior_Edu_Postcode' column by:
    - Converting values to strings and stripping spaces
    - Replacing any zero or asterisk values (even with spaces) with '0000_ABROAD'
    - Adding underscores between words in postcode strings
    - Replacing rare postcodes (frequency < threshold) with 'Others'
    """
    # Convert to string and strip
    data['Prior_Edu_Postcode'] = data['Prior_Edu_Postcode'].astype(str).str.strip()
    
    # Replace zeros or asterisks with '0000_ABROAD' using regex
    data['Prior_Edu_Postcode'] = data['Prior_Edu_Postcode'].replace(
        to_replace=[r'^0+$', r'^\*+$'], value='0000_ABROAD', regex=True
    )
    
    # Add underscores between words
    data['Prior_Edu_Postcode'] = data['Prior_Edu_Postcode'].apply(lambda x: "_".join(x.split()))
    
    # Replace rare values with 'Others'
    value_counts = data['Prior_Edu_Postcode'].value_counts()
    small_values = value_counts[value_counts < threshold].index
    data['Prior_Edu_Postcode'] = data['Prior_Edu_Postcode'].replace(small_values, replacement)
    
    return data

# ✅ Validator function
def validate_prior_edu_postcode(data):
    """
    Check for leftover zero, asterisk, or trailing whitespace issues in Prior_Edu_Postcode.
    """
    remaining_issues = data['Prior_Edu_Postcode'][data['Prior_Edu_Postcode'].str.match(r'^(0+|\*+)$')]
    if remaining_issues.empty:
        print("✅ No '0' or '*' values remain in Prior_Edu_Postcode!")
    else:
        print("⚠️ Still found problematic values:")
        print(remaining_issues.value_counts())
    
    # Check for trailing spaces (should not happen if cleaned, but good check)
    trailing_space_issues = data['Prior_Edu_Postcode'][data['Prior_Edu_Postcode'].str.contains(r'\s$')]
    if trailing_space_issues.empty:
        print("✅ No trailing spaces found in Prior_Edu_Postcode.")
    else:
        print("⚠️ Found trailing spaces in:")
        print(trailing_space_issues.unique())

# ✅ Example Usage:
data = clean_prior_edu_postcode(data)   # Clean the column
validate_prior_edu_postcode(data)       # Validate if everything is fixed
data['Prior_Edu_Postcode'].value_counts()  # Check distribution if desired

In [ ]:
data.columns

In [ ]:
print(data['Prior_Edu_School_Location'].value_counts())

In [ ]:
# Function to clean Prior_Edu_School_Location
def clean_prior_edu_school_location(data, threshold=50, abroad_replacement="0000_ABROAD", rare_replacement="Others"):
    """
    Cleans the 'Prior_Edu_School_Location' column by:
    - Stripping spaces
    - Replacing values '0', '*', or '-1' with abroad_replacement
    - Adding underscores between words
    - Replacing rare values (below threshold) with rare_replacement
    """
    # Strip spaces
    data['Prior_Edu_School_Location'] = data['Prior_Edu_School_Location'].astype(str).str.strip()
    
    # Replace '0', '*', or '-1' (even if they appear alone) with abroad_replacement
    data['Prior_Edu_School_Location'] = data['Prior_Edu_School_Location'].replace(
        to_replace=[r'^0+$', r'^\*+$', r'^-1$'], value=abroad_replacement, regex=True
    )
    
    # Add underscores between words
    data['Prior_Edu_School_Location'] = data['Prior_Edu_School_Location'].apply(lambda x: "_".join(x.split()))
    
    # Replace values appearing less than threshold times with rare_replacement
    value_counts = data['Prior_Edu_School_Location'].value_counts()
    small_values = value_counts[value_counts < threshold].index
    data['Prior_Edu_School_Location'] = data['Prior_Edu_School_Location'].replace(small_values, rare_replacement)
    
    return data

def validate_prior_edu_school_location(data):
    """
    Validates that no invalid values ('0', '*', '-1') or trailing spaces remain in 'Prior_Edu_School_Location'.
    """
    issues = data['Prior_Edu_School_Location'][data['Prior_Edu_School_Location'].str.match(r'^(0+|\*+|-1)$')]
    if issues.empty:
        print("✅ No invalid '0', '*', or '-1' values remain in Prior_Edu_School_Location.")
    else:
        print("⚠️ Found invalid values:")
        print(issues.value_counts())
    
    trailing_spaces = data['Prior_Edu_School_Location'][data['Prior_Edu_School_Location'].str.contains(r'\s$')]
    if trailing_spaces.empty:
        print("✅ No trailing spaces found.")
    else:
        print("⚠️ Found trailing spaces in:")
        print(trailing_spaces.unique())

# ✅ Usage:
data = clean_prior_edu_school_location(data)
validate_prior_edu_school_location(data)
data['Prior_Edu_School_Location'].value_counts()

Engineer Prior_Edu_Country

In [ ]:
data['Prior_Edu_Country'].value_counts()

In [ ]:
def clean_prior_edu_country(data, threshold=50, abroad_replacement="outside_NL", rare_replacement="Others"):
    """
    Cleans the 'Prior_Edu_Country' column by:
    - Stripping spaces
    - Replacing '0' and '-1' with abroad_replacement
    - Adding underscores between words
    - Replacing rare values (below threshold) with rare_replacement
    """
    # Strip spaces
    data['Prior_Edu_Country'] = data['Prior_Edu_Country'].astype(str).str.strip()
    
    # Replace '0' and '-1' with abroad_replacement
    data['Prior_Edu_Country'] = data['Prior_Edu_Country'].replace(
        to_replace=[r'^0+$', r'^-1$'], value=abroad_replacement, regex=True
    )
    
    # Add underscores between words
    data['Prior_Edu_Country'] = data['Prior_Edu_Country'].apply(lambda x: "_".join(x.split()))
    
    # Replace rare values
    value_counts = data['Prior_Edu_Country'].value_counts()
    small_values = value_counts[value_counts < threshold].index
    data['Prior_Edu_Country'] = data['Prior_Edu_Country'].replace(small_values, rare_replacement)
    
    return data

def validate_prior_edu_country(data):
    """
    Validates that no invalid values ('0' or '-1') or trailing spaces remain in 'Prior_Edu_Country'.
    """
    issues = data['Prior_Edu_Country'][data['Prior_Edu_Country'].str.match(r'^(0+|-1)$')]
    if issues.empty:
        print("✅ No invalid '0' or '-1' values remain in Prior_Edu_Country.")
    else:
        print("⚠️ Found invalid values:")
        print(issues.value_counts())
    
    trailing_spaces = data['Prior_Edu_Country'][data['Prior_Edu_Country'].str.contains(r'\s$')]
    if trailing_spaces.empty:
        print("✅ No trailing spaces found.")
    else:
        print("⚠️ Found trailing spaces in:")
        print(trailing_spaces.unique())

# ✅ Usage:
data = clean_prior_edu_country(data)
validate_prior_edu_country(data)
data['Prior_Edu_Country'].value_counts()

In [ ]:
data.columns

In [ ]:
df1 = data
data = df1

# Feature Extraction

In [ ]:
# Create Student_Enrollment_Gap column from Prior_Edu_Exam_Date and Student_Start_Date

# Ensure the date columns are in datetime format
data['Student_Start_Date'] = pd.to_datetime(data['Student_Start_Date'], errors='coerce')
data['Prior_Edu_Exam_Date'] = pd.to_datetime(data['Prior_Edu_Exam_Date'], errors='coerce')

# Calculate the gap in days, then convert it to weeks and round down the result
data['Student_Enrollment_Gap'] = np.floor((data['Student_Start_Date'] - data['Prior_Edu_Exam_Date']).dt.days / 7)

# Optional: To handle cases where the date conversion fails (NaT), you can fill them with a value, e.g., 0 or another placeholder
data['Student_Enrollment_Gap'].fillna(0, inplace=True)

# Check the result
data[['Student_Start_Date', 'Prior_Edu_Exam_Date', 'Student_Enrollment_Gap']].head()

In [ ]:
# Could negative values indicate that some students starting in HU before their Prior_Edu_Exam_Date? Could this be true or its not possible.
data['Student_Enrollment_Gap'].value_counts()

In [ ]:
# Could negative values indicate that some students starting in HU before their Prior_Edu_Exam_Date? Could this be true or its not possible.
data['Student_Enrollment_Gap'].value_counts()

In [ ]:
print(data['Student_Enrollment_Gap'].nlargest(100))


In [ ]:
# Clean and preprocess the Student_Enrollment_Gap column by removing negative signs, 
# capping values over 522 weeks (10 years), and imputing them using KNN (K=5) to ensure 
# all values stay within a valid range.

# Step 1: Remove minus signs and convert to numeric
data['Student_Enrollment_Gap'] = data['Student_Enrollment_Gap'].astype(str).str.replace('-', '', regex=False)
data['Student_Enrollment_Gap'] = pd.to_numeric(data['Student_Enrollment_Gap'], errors='coerce')

# Step 2: Replace values > 522 with NaN (so we can impute them)
data.loc[data['Student_Enrollment_Gap'] > 522, 'Student_Enrollment_Gap'] = np.nan

# Step 3: Use KNN Imputer with K=5
imputer = KNNImputer(n_neighbors=5)
data[['Student_Enrollment_Gap']] = imputer.fit_transform(data[['Student_Enrollment_Gap']])

# Step 4: Final check to make sure no value exceeds 522
data.loc[data['Student_Enrollment_Gap'] > 522, 'Student_Enrollment_Gap'] = 522


In [ ]:
data['Student_Enrollment_Gap'].value_counts()

In [ ]:
print(data['Student_Enrollment_Gap'].nlargest(10))

In [ ]:
# Reset cells output
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')
pd.reset_option('display.width')
pd.reset_option('display.expand_frame_repr')

In [ ]:
data.columns

Create Exit_Status common from 'Graduating', 'Dropping', and 'Switching'

In [ ]:
data.info()

In [ ]:
data[['Graduating']].value_counts()

In [ ]:
data[['Dropping']].value_counts()

In [ ]:
data[['Switching']].value_counts()

In [ ]:
# Step 1: Create the 'Exit_Status' column and set default to 'Dropping'
data['Exit_Status'] = 'dropping'  

# Step 2: Assign 'Graduating' where graduating == 1
data.loc[data['Graduating'] == 1, 'Exit_Status'] = 'graduating'  

# Step 3: Assign 'switching' where switching == 1
data.loc[data['Switching'] == 1, 'Exit_Status'] = 'switching'  

# Step 4: Drop the original columns since they are now redundant
data.drop(columns=['Graduating', 'Dropping', 'Switching'], inplace=True)  

# Step 5: Verify the results
print(data['Exit_Status'].value_counts())


In [ ]:
# Save version 1 data here to later combine it with version 2
data.to_csv(os.path.join("..", "data", "processed", "dropout_version1_data.csv"), header=True)